# Residual Stream Decomposition

Fine-grained decomposition of the residual stream into per-head, per-MLP,
and embedding contributions. Includes direction projection, orthogonal
decomposition, and interference analysis.

Reference: Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"

## Why This Matters

Composition residual stream decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_stream_decomposition import (
    per_head_residual_decomposition,
    direction_projection_analysis,
    orthogonal_decomposition,
    token_prediction_decomposition,
    component_interference_decomposition,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
print("Model ready")

## Per-Head Residual Decomposition

Decompose the final residual into individual head contributions,
MLP contributions, and the embedding.

In [ ]:
result = per_head_residual_decomposition(model, tokens)
print(f"Head norms:")
for l in range(cfg.n_layers):
    norms = [f"{result['head_norms'][l, h]:.3f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {norms}")
print(f"\nMLP contribution norms:")
for l, c in result['mlp_contributions'].items():
    print(f"  Layer {l}: {np.linalg.norm(c):.4f}")
print(f"\nEmbedding norm: {np.linalg.norm(result['embedding_contribution']):.4f}")

## Direction Projection Analysis

Project each component's contribution onto a target direction to see
which components promote a specific concept.

In [ ]:
# Use the unembedding direction for token 0 as target
W_U = np.array(model.unembed.W_U)
direction = W_U[:, 0]
result = direction_projection_analysis(model, tokens, direction)
print(f"Total projection: {result['total_projection']:.4f}")
print(f"Embedding projection: {result['embedding_projection']:.4f}")
print(f"\nTop contributors:")
for name, proj in result['top_contributors'][:10]:
    sign = '+' if proj > 0 else ''
    print(f"  {name}: {sign}{proj:.4f}")

## Orthogonal Decomposition

Find an orthogonal basis from component contributions that explains
the residual stream variance.

In [ ]:
result = orthogonal_decomposition(model, tokens)
print(f"Total variance: {result['total_variance']:.4f}")
print(f"Number of basis vectors: {len(result['basis_vectors'])}")
print(f"\nCumulative explained variance:")
for i, (label, cum) in enumerate(zip(result['basis_labels'][:10], result['cumulative_explained'][:10])):
    print(f"  {i+1}. {label}: {cum:.1%}")

## Token Prediction Decomposition

Decompose the logit of a specific target token into per-component contributions.

In [ ]:
result = token_prediction_decomposition(model, tokens, target_token=5)
print(f"Total logit for token 5: {result['total_logit']:.4f}")
print(f"Embedding logit: {result['embedding_logit']:.4f}")
print(f"Bias logit: {result['bias_logit']:.4f}")
print(f"\nTop boosters:")
for name, val in result['top_positive'][:5]:
    print(f"  {name}: +{val:.4f}")
print(f"\nTop suppressors:")
for name, val in result['top_negative'][:5]:
    print(f"  {name}: {val:.4f}")

## Component Interference

Analyze constructive and destructive interference between components.

In [ ]:
result = component_interference_decomposition(model, tokens)
print(f"Mean alignment: {result['mean_alignment']:.4f}")
print(f"\nConstructive pairs (aligned):")
for a, b, align in result['constructive_pairs'][:5]:
    print(f"  {a} <-> {b}: {align:.4f}")
print(f"\nDestructive pairs (opposed):")
for a, b, align in result['destructive_pairs'][:5]:
    print(f"  {a} <-> {b}: {align:.4f}")